|<img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">| <p> Diplomado en Inteligencia Artificial para los Negocios </p> <p> Facultad de Ingeniería y Negocios </p> <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p> <p>Semana 00: Smoke test del entorno</p>|
|:--- |:---|


# 00 · Smoke test del entorno

Valida que **PyCaret 4.0.0a8** (alpha) y **StatsForecast** corren bien en Colab *antes* de construir las 8 clases encima.

Si todas las celdas terminan sin error, el entorno está listo.

## 1. Instalación (versión pineada)

PyCaret 4 es alpha: fijamos la versión exacta para que no cambie sola.

In [ ]:
# Versión exacta para reproducibilidad del curso
!pip install -q pycaret==4.0.0a8 statsforecast

> **Si hay conflictos de versiones** (NumPy 2 / pandas 2 / scikit-learn 1.7), ve a **Runtime → Restart session** y vuelve a correr desde la celda de imports (sin reinstalar).

## 2. Verificación de versiones

In [ ]:
import pycaret, statsforecast
print('pycaret      :', pycaret.__version__)   # esperado: 4.0.0a8
print('statsforecast:', statsforecast.__version__)

## 3. Prueba mínima — PyCaret 4 (API nueva)

API v4: se importa desde `pycaret.tasks`, se instancia un `Experiment` con la config y se llama `.fit(df)`. **No** se usa el `setup()` funcional de la v3.

In [ ]:
from pycaret.datasets import get_data
from pycaret.tasks import ClassificationExperiment

df = get_data('juice')
exp = ClassificationExperiment(target='Purchase', session_id=42).fit(df)
best = exp.compare_models().best   # compare_models() devuelve un objeto; el modelo va en .best
print('Mejor modelo:', best)
exp.save_model(best, 'smoke_model')

## 4. Prueba mínima — StatsForecast

Formato largo (`unique_id`, `ds`, `y`) y `freq` explícita. Incluimos baselines (`Naive`, `SeasonalNaive`) junto al modelo automático.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive, Naive
from statsforecast.utils import AirPassengersDF

df_ts = AirPassengersDF
sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=12), AutoARIMA(season_length=12)],
    freq='ME',
)
sf.fit(df_ts)
forecast = sf.predict(h=12, level=[95])
forecast.head()

## 5. Validación temporal (rolling-origin)

Así se evalúa respetando el orden del tiempo — clave para la clase 7.

In [ ]:
# En statsforecast 2.x, cross_validation recibe el df explícito (no usa el estado de fit)
cv = sf.cross_validation(df=df_ts, h=12, n_windows=3, step_size=12)
cv.head()

---
✅ **Si llegaste hasta aquí sin errores, el entorno está validado.**

Recordatorio: PyCaret 4 es alpha. Si una clase o método de `pycaret.tasks` no coincide, ajusta contra la doc oficial (pycaret.org/docs), no contra blogs de la v3.